In [ ]:
import os
import re
import torch
import datetime
import hashlib
import librosa
import numpy as np
import soundfile as sf
import wave
import weave
from huggingface_hub import snapshot_download, login
from pathlib import Path
from qwen_tts import Qwen3TTSModel
from IPython.display import Audio
from openai import OpenAI
from dotenv import load_dotenv
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Wav2Vec2FeatureExtractor, WavLMForXVector

load_dotenv(override=True)

In [ ]:
#client = OpenAI()

In [ ]:
input_audio_dir = Path("./audio/input/")

In [ ]:
HF_TOKEN = os.environ.get('HUGGINGFACEHUB_API_TOKEN')
login(token=HF_TOKEN)

In [ ]:
weave.init(f"{os.getenv("WANDB_ACCOUNT")}/{os.getenv("WEAVE_PROJECT")}")

# Manual Run

In [ ]:
def get_model_path(model_type: str, model_size: str) -> str:
    """Get model path based on type and size."""
    return snapshot_download(f"Qwen/Qwen3-TTS-12Hz-{model_size}-{model_type}")

def _normalize_audio(x, eps=1e-12, clip=True):
    """Normalize audio to float32 in [-1, 1] range."""
    # x = np.asarray(wav)

    if np.issubdtype(x.dtype, np.integer):
        info = np.iinfo(x.dtype)
        if info.min < 0:
            y = x.astype(np.float32) / max(abs(info.min), info.max)
        else:
            mid = (info.max + 1) / 2.0
            y = (x.astype(np.float32) - mid) / mid
    elif np.issubdtype(x.dtype, np.floating):
        y = x.astype(np.float32)
        m = np.max(np.abs(y)) if y.size else 0.0
        if m > 1.0 + 1e-6:
            y = y / (m + eps)
    else:
        raise TypeError(f"Unsupported dtype: {x.dtype}")

    if clip:
        y = np.clip(y, -1.0, 1.0)

    if y.ndim > 1:
        y = np.mean(y, axis=-1).astype(np.float32)

    return y

def write_sound_to_file(
    output_filename: str,
    audio_file: np.array,
    sample_rate: int,
    ) -> None:

    # Write the file
    # subtype='PCM_16' ensures it's a standard WAV file compatible with all players
    sf.write(
      output_filename,
      audio_file,
      sample_rate,
      subtype='PCM_16',
      )

def wavfile_to_array(soundfile):

    # Extract raw parameters
    sample_rate = soundfile.getframerate()
    n_channels = soundfile.getnchannels()
    n_frames = soundfile.getnframes()

    # Read all frames as bytes
    raw_bytes = soundfile.readframes(n_frames)

    # 2. Convert raw bytes to NumPy array
    # Standard WAV is 16-bit integer, so we use int16
    # If your audio sounds like static, the dtype might need to be np.uint8 or np.int32
    audio_data = np.frombuffer(raw_bytes, dtype=np.int16)

    # 3. Handle Stereo (Multiple Channels)
    # The wave library interleaves channels (L, R, L, R...), so we reshape
    if n_channels > 1:
        audio_data = audio_data.reshape(-1, n_channels)
        # If the player expects (channels, samples), transpose it:
        audio_data = audio_data.T

    return audio_data, sample_rate

In [ ]:
# @weave.op(call_display_name=lambda call: f"{call.func_name}-{int(datetime.datetime.now().timestamp())}")
# def transcribe_audio_from_file(filename: str, model: str = "gpt-4o-transcribe") -> str:
#   audio_file = open(filename, "rb")

#   transcription = client.audio.transcriptions.create(
#       model=model,
#       file=audio_file,
#       response_format="text"
#       )

#   return transcription

@weave.op(call_display_name=lambda call: f"{call.func_name}-{int(datetime.datetime.now().timestamp())}")
def transcribe_audio_from_file(filename: str, model_name: str = "openai/whisper-small") -> str:

    processor = WhisperProcessor.from_pretrained(model_name)
    model = WhisperForConditionalGeneration.from_pretrained(model_name)
    model.config.forced_decoder_ids = None

    audio, sr = librosa.load(filename, sr=16000)

    input_features = processor(audio, sampling_rate=sr, return_tensors="pt").input_features

    # generate token ids
    predicted_ids = model.generate(language="en", input_features=input_features)

    # decode token ids to text
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
 
    return transcription[0]

@weave.op(call_display_name=lambda call: f"{call.func_name}-{int(datetime.datetime.now().timestamp())}")
def generate_voice_clone(
    ref_audio,
    ref_text,
    target_text,
    language,
    use_xvector_only,
    max_new_tokens=2048,
    ):

    """Generate speech using Base (Voice Clone) model."""
    if not target_text or not target_text.strip():
        return None, "Error: Target text is required."

    audio_tuple = ref_audio
    if audio_tuple is None:
        return None, "Error: Reference audio is required."

    if not use_xvector_only and (not ref_text or not ref_text.strip()):
        return None, "Error: Reference text is required when 'Use x-vector only' is not enabled."

    try:
        wavs, sr = voice_clone_model.generate_voice_clone(
            text=target_text.strip(),
            language=language,
            ref_audio=audio_tuple,
            ref_text=ref_text.strip() if ref_text else None,
            x_vector_only_mode=use_xvector_only,
            max_new_tokens=max_new_tokens,
        )
        print("Voice clone generation completed successfully!")

        generated_audio_dir = Path("./audio/cloned_audio/")
        generated_audio_dir.mkdir(parents=True, exist_ok=True)
        output_filename = generated_audio_dir / "generated_cloned_audio.wav"

        # Write the output to file
        write_sound_to_file(
            output_filename = output_filename,
            audio_file = wavs[0],
            sample_rate = sr,
            )

        print(f"Voice Clone generation written to file {str(output_filename)}")

    except Exception as e:
        return None, f"Error: {type(e).__name__}: {e}"

    # Return the file to Weave op
    return {"generated_audio": wave.open(str(output_filename), "rb"), "sample_rate": sr, "audio_array": wavs[0]}

@weave.op(call_display_name=lambda call: f"{call.func_name}-{int(datetime.datetime.now().timestamp())}")
def clone_voice(
    input_audio_filename: str,
    clone_target_text: str,
    max_new_tokens: int = 2048,
    ):

  # Transcribe Audio
  print("Transcribing Audio")
  transcribed_audio = transcribe_audio_from_file(input_audio_filename)
  print(f"Transcribed Audio: {transcribed_audio}")

  # Extract Audio Array
  print("Extracting Audio Array")
  audio_data, sample_rate = librosa.load(input_audio_filename, sr=None)

  # Generate Voice - output is sample rate and audio file array
  # Needs to be a numpy array
  clone_ref_audio = (_normalize_audio(audio_data), sample_rate)

  # Transcript of Reference Audio
  clone_ref_text: str = transcribed_audio.strip()
  clone_xvector: bool = False
  clone_language: str = "Auto"

  print("Cloning Audio")

  generated_clone_voice_output = generate_voice_clone(
    ref_audio=clone_ref_audio,
    ref_text=clone_ref_text,
    target_text=clone_target_text,
    language=clone_language,
    use_xvector_only=clone_xvector,
    max_new_tokens=max_new_tokens,
  )

  # Return the file to Weave op
  return {"input_audio": wave.open(str(input_audio_filename)), **generated_clone_voice_output}

In [ ]:
# Load the Voice Clone Model
try:
  print(voice_clone_model)
except:
  voice_clone_model = Qwen3TTSModel.from_pretrained(
      get_model_path("Base", "1.7B"),
      device_map=torch.device("cuda") if torch.cuda.is_available() else torch.device("mps"),
      dtype=torch.bfloat16,
      token=HF_TOKEN,
      #attn_implementation="kernels-community/flash-attn3",
      )

In [ ]:
clone_target_text: str = """
Batman here... I use Weights and Biases to help me watch over Gotham City. Whether it's the Joker, the penguin or Two-Face, Weights and Biases has you covered. 
""".strip()

cloned_voice_output = clone_voice(
    input_audio_filename = input_audio_dir / "batman_original.wav", 
    clone_target_text = clone_target_text,
    )

In [ ]:
cloned_voice_output

In [ ]:
# Original Audio
audio_path = input_audio_dir / "batman_original.wav"
audio_data, sample_rate = librosa.load(audio_path, sr=None)
display(
    Audio(
        audio_data,
        rate=sample_rate,
        autoplay=False
        )
)

# Generated Audio
display(
    Audio(
        cloned_voice_output["audio_array"],
        rate=cloned_voice_output["sample_rate"],
        autoplay=False
        )
    )

## Batch Voice Cloning from Same Ref Voice

In [ ]:
# Voice Clone Prompt for multiple generations
prompt_items = voice_clone_model.create_voice_clone_prompt(
    ref_audio=str(input_audio_dir / "batman_original.wav"),
    ref_text="The Lazarus Pit has corrupted your mind. Think about it. If your pit falls into the wrong hands, you'll be powerless to stop centuries of destruction. This is your chance for redemption, Raish. Call off your crusade, or I'll be back for you.",
    x_vector_only_mode=False,
)


In [ ]:
wavs, sr = voice_clone_model.generate_voice_clone(
    text=["It's over Joker!", "You won't get away with this! I'll stop you, no matter what!"],
    language=["English", "English"],
    voice_clone_prompt=prompt_items,
    x_vector_only_mode=[False, False],
    max_new_tokens=2096,
    do_sample=False,
)

In [ ]:
prompt_items[0].ref_spk_embedding

In [ ]:
display(
    Audio(
        wavs[1],
        rate=sr,
        autoplay=False
        )
    )

In [ ]:
sr

# Use Weave Model Class

In [ ]:
from typing import Any
from pydantic import PrivateAttr
from weave import Model

In [ ]:
def _drop_clone_audio_array(output):
    """Hide audio_array from the Weave trace; runtime return value is unchanged."""
    if isinstance(output, dict) and "audio_array" in output:
        return {k: v for k, v in output.items() if k != "audio_array"}
    return output


def _open_input_audio_for_trace(inputs):
    """Replace the input_audio string path with an opened wave file so Weave
    renders it as a playable audio widget in the trace (same as generated_audio)."""
    path = inputs.get("input_audio")
    if isinstance(path, str):
        try:
            return {**inputs, "input_audio": wave.open(path, "rb")}
        except Exception:
            return inputs
    return inputs

@weave.op(call_display_name=lambda call: f"{call.func_name}-{int(datetime.datetime.now().timestamp())}")
def transcribe_audio_from_file(filename: str, model_name: str = "openai/whisper-small") -> str:

    processor = WhisperProcessor.from_pretrained(model_name)
    model = WhisperForConditionalGeneration.from_pretrained(model_name)
    model.config.forced_decoder_ids = None

    audio, sr = librosa.load(filename, sr=16000)

    input_features = processor(audio, sampling_rate=sr, return_tensors="pt").input_features

    # generate token ids
    predicted_ids = model.generate(language="en", input_features=input_features)

    # decode token ids to text
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
 
    return transcription[0]

class VoiceCloneModel(Model):
    model_size: str
    model_type: str

    # Lazy-loaded Qwen3TTSModel weights. PrivateAttr → not validated, not serialized
    # into the Weave Model version, not shown as an input on traces.
    _qwen_model: Any = PrivateAttr(default=None)

    # Single-slot cache for the voice-clone prompt and transcript. Identity is
    # a sha256 of the reference-audio bytes — the marimo app overwrites a stable
    # path (`audio/input/_uploaded.{ext}`) on every upload, so path alone can't
    # distinguish between different references. Content-hashing the file bytes
    # is cheap (ms for typical clips) and content-correct.
    _cache_key: Any = PrivateAttr(default=None)         # (content_sha256, ref_text_arg)
    _cached_transcript: Any = PrivateAttr(default=None) # str — auto-transcribed or override
    _cached_prompt: Any = PrivateAttr(default=None)     # list of prompt_items

    def _get_model_path(self, model_type: str, model_size: str) -> str:
        """Get model path based on type and size."""
        return snapshot_download(f"Qwen/Qwen3-TTS-12Hz-{model_size}-{model_type}")

    def _ensure_loaded(self):
        """Load Qwen3TTSModel on first use; reuse for the rest of the session."""
        if self._qwen_model is None:
            print(f"Loading Qwen3-TTS-12Hz-{self.model_size}-{self.model_type} (first call only)...")
            self._qwen_model = Qwen3TTSModel.from_pretrained(
                self._get_model_path(self.model_type, self.model_size),
                device_map=torch.device("cuda") if torch.cuda.is_available() else torch.device("mps"),
                dtype=torch.bfloat16,
                token=os.environ.get("HUGGINGFACEHUB_API_TOKEN"),
            )
        return self._qwen_model

    def _normalize_audio(self, x, eps=1e-12, clip=True):
        """Normalize audio to float32 in [-1, 1] range."""
        if np.issubdtype(x.dtype, np.integer):
            info = np.iinfo(x.dtype)
            if info.min < 0:
                y = x.astype(np.float32) / max(abs(info.min), info.max)
            else:
                mid = (info.max + 1) / 2.0
                y = (x.astype(np.float32) - mid) / mid
        elif np.issubdtype(x.dtype, np.floating):
            y = x.astype(np.float32)
            m = np.max(np.abs(y)) if y.size else 0.0
            if m > 1.0 + 1e-6:
                y = y / (m + eps)
        else:
            raise TypeError(f"Unsupported dtype: {x.dtype}")

        if clip:
            y = np.clip(y, -1.0, 1.0)
        if y.ndim > 1:
            y = np.mean(y, axis=-1).astype(np.float32)
        return y

    def _write_sound_to_file(self, output_filename, audio_file, sample_rate):
        sf.write(output_filename, audio_file, sample_rate, subtype="PCM_16")

    def _audio_content_key(self, input_audio: str) -> str:
        """Fingerprint the reference audio by hashing its raw bytes. Cheap
        (milliseconds for typical clips) and content-based, so two uploads
        that overwrite the same path with different audio produce different
        keys → correct cache miss."""
        return hashlib.sha256(Path(input_audio).read_bytes()).hexdigest()

    @weave.op(
        call_display_name="build_voice_clone_prompt",
        postprocess_inputs=_open_input_audio_for_trace,
    )
    def _build_voice_clone_prompt(self, input_audio: str, ref_text: str):
        """Encode the reference clip into prompt_items. Surfaced as its own
        @weave.op so cache hits/misses are visible in the trace tree: the op
        fires under predict on a miss, and is absent on a hit. The input_audio
        path is wrapped as an opened wave file by `_open_input_audio_for_trace`
        so Weave renders a playable widget instead of a bare path string."""
        return self._ensure_loaded().create_voice_clone_prompt(
            ref_audio=input_audio,
            ref_text=ref_text,
            x_vector_only_mode=False,
        )

    @weave.op(
        call_display_name="VoiceClone",
        postprocess_inputs=_open_input_audio_for_trace,
        postprocess_output=_drop_clone_audio_array,
    )
    def predict(
        self,
        input_audio: str,
        target_text: str,
        max_new_tokens: int = 2048,
        ref_text: str | None = None,
        **kwargs,
    ):
        if not target_text or not target_text.strip():
            return None, "Error: Target text is required."

        # Identity is the audio's content hash, not its path. ref_text_arg
        # is also keyed so flipping between auto-transcribe (None) and a
        # manual override (string) doesn't return a stale prompt.
        content_hash = self._audio_content_key(input_audio)
        cache_key = (content_hash, ref_text)

        if self._cache_key == cache_key:
            print(
                f"[VoiceClone] Cache HIT for {input_audio!r} "
                f"(content={content_hash[:12]}…, ref_text={'override' if ref_text else 'auto'}). "
                "Reusing cached transcript + voice-clone prompt."
            )
            transcript = self._cached_transcript
            prompt = self._cached_prompt
        else:
            print(
                f"[VoiceClone] Cache MISS for {input_audio!r} "
                f"(content={content_hash[:12]}…, ref_text={'override' if ref_text else 'auto'})."
            )
            if ref_text is None:
                print("[VoiceClone]   → Transcribing reference audio with Whisper…")
                transcript = transcribe_audio_from_file(input_audio).strip()
            else:
                print("[VoiceClone]   → Using caller-supplied reference transcript.")
                transcript = ref_text.strip()
            print("[VoiceClone]   → Building voice-clone prompt (encoding speaker embedding)…")
            prompt = self._build_voice_clone_prompt(input_audio, transcript)
            self._cache_key = cache_key
            self._cached_transcript = transcript
            self._cached_prompt = prompt
            print("[VoiceClone]   → Cache populated.")

        print("[VoiceClone] Generating target audio…")
        try:
            wavs, sr = self._ensure_loaded().generate_voice_clone(
                text=[target_text.strip()],
                language=["Auto"],
                voice_clone_prompt=prompt,
                x_vector_only_mode=[False],
                max_new_tokens=max_new_tokens,
                **kwargs,
            )
        except Exception as e:
            return None, f"Error: {type(e).__name__}: {e}"

        generated_audio_dir = Path("./audio/cloned_audio/")
        generated_audio_dir.mkdir(parents=True, exist_ok=True)
        output_filename = generated_audio_dir / "generated_cloned_audio.wav"
        self._write_sound_to_file(
            output_filename=output_filename,
            audio_file=wavs[0],
            sample_rate=sr,
        )
        print(f"[VoiceClone] Generation complete → {output_filename}")

        return {
            "generated_audio": wave.open(str(output_filename), "rb"),
            "sample_rate": sr,
            "audio_array": wavs[0],
        }


In [ ]:
try:
    print(voice_clone_model_op)
except NameError:
    voice_clone_model_op = VoiceCloneModel(model_size="1.7B", model_type="Base")

input_audio_path = str(input_audio_dir / "batman_original.wav")
target_text = (
    "Batman here... I use Weights and Biases to help me watch over Gotham City. "
    "Whether it's the Joker, the penguin or Two-Face, Weights and Biases has you covered."
)

cloned_voice_output = voice_clone_model_op.predict(
    input_audio=input_audio_path,
    target_text=target_text,
    max_new_tokens=2048,
)

In [ ]:
# Generated Audio
display(
    Audio(
        cloned_voice_output["audio_array"],
        rate=cloned_voice_output["sample_rate"],
        autoplay=False,
    )
)
